In [2]:
from physics.simulation import mcfm, msq
from physics.analysis import zz4l
from physics.hstar import c6, eft

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt

In [15]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw  = 1.5

In [4]:
filenames = {
    msq.Component.SBI : 'data/zz4l/ggZZ_sbi/analyzed.csv',
    msq.Component.SIG : 'data/zz4l/ggZZ_sig/analyzed.csv',
}

In [27]:
events_sbi = mcfm.from_csv(file_path=filenames[msq.Component.SBI], kinematics=['4l_mass', 'l1_pt'])
events_sig = mcfm.from_csv(file_path=filenames[msq.Component.SIG], kinematics=['4l_mass', 'l1_pt'])

In [6]:
from physics.constants import cH_to_c6, ctH_to_ct, cHG_to_cg, fb_to_ab

cH_val = 50
c6_val = cH_val * cH_to_c6

ctH_val = 2.0
ct_val = ctH_val * ctH_to_ct

cHG_val = 0.02
cg_val = cHG_val * cHG_to_cg

# mod_sig = c6.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_points = [-20, -10, 0, 10, +20])
# wt_sig_c6, prob_sig_c6 = mod_sig.modify(c6_values = [c6_val, -c6_val])
mod_sig = eft.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_points = [-20, -10, 0, +10, +20])
wt_sig_c6, prob_sig_c6 = mod_sig.modify(c6 = [c6_val, -c6_val], ct = None, cg = None)
wt_sig_ct, prob_sig_ct = mod_sig.modify(c6 = None, ct = [ct_val, -ct_val], cg = None)
wt_sig_cg, prob_sig_cg = mod_sig.modify(c6 = None, ct = None, cg = [cg_val, -cg_val])

# mod_sbi = c6.Modifier(baseline=msq.Component.SBI, events=events_sbi)
# wt_sbi_c6, prob_sbi_c6 = mod_sbi.modify(c6_values = [c6_val, -c6_val])
mod_sbi = eft.Modifier(baseline=msq.Component.SBI, events=events_sbi)
wt_sbi_c6, prob_sbi_c6 = mod_sbi.modify(c6 = [c6_val, -c6_val], ct = None, cg = None)
wt_sbi_ct, prob_sbi_ct = mod_sbi.modify(c6 = None, ct = [ct_val, -ct_val], cg = None)
wt_sbi_cg, prob_sbi_cg = mod_sbi.modify(c6 = None, ct = None, cg = [cg_val, -cg_val])

In [7]:
sm_color = 'black'
sm_line = '--'

c6_vars   = [c6_val, -c6_val]
cH_vars   = [cH_val, -cH_val]
c6_color  = 'red'

ct_vars   = [ct_val, -ct_val]
ctH_vars  = [ctH_val, -ctH_val]
ct_color  = 'blue'

cg_vars   = [cg_val, -cg_val]
cHG_vars  = [cHG_val, -cHG_val]
cg_color  = 'green'

bsm_lines  = ['-', ':']

In [100]:
xobs_sig = events_sig.kinematics['4l_mass'].to_numpy()
xobs_sbi = events_sbi.kinematics['4l_mass'].to_numpy()

nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)
xname = 'mzz'
xlabel = 'm_{ZZ}'

ymin_sig, ymax_sig = 0.0, 0.0015 * fb_to_ab
ymin_sbi, ymax_sbi = 1e-5, 0.1

In [84]:
xobs_sig = events_sig.kinematics['l1_pt'].to_numpy()
xobs_sbi = events_sbi.kinematics['l1_pt'].to_numpy()

nxbins = 38
xmin, xmax = 20.0, 400.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)
xname = 'ptl1'
xlabel = 'p_{\\mathrm{T}}^{\ell_1}'

ymin_sig, ymax_sig = 0.0, 0.004 * fb_to_ab
ymin_sbi, ymax_sbi = 1e-5, 1.0

In [101]:
h_sig_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig_sm.fill(xobs_sig, weight=events_sig.weights)

h_sig_c6 = []
for i, c6_var in enumerate(c6_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_c6[:,i])
    h_sig_c6.append(h)

h_sig_ct = []
for i, ct_var in enumerate(ct_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_ct[:,i])
    h_sig_ct.append(h)

h_sig_cg = []
for i, cg_var in enumerate(cg_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_cg[:,i])
    h_sig_cg.append(h)

h_sbi_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi_sm.fill(xobs_sbi, weight=events_sbi.weights)

h_sbi_c6 = []
for i, c6_var in enumerate(c6_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_c6[:,i])
    h_sbi_c6.append(h)

h_sbi_ct = []
for i, ct_var in enumerate(ct_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_ct[:,i])
    h_sbi_ct.append(h)

h_sbi_cg = []
for i, cg_var in enumerate(cg_vars):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_cg[:,i])
    h_sbi_cg.append(h)

In [92]:
fig, (ax_main, ax_ratio) = plt.subplots(2,1, gridspec_kw={'height_ratios' : [3,1]}, figsize=(5,5), sharex=True, sharey=False)

# SIG

l_c6 = []
for i, c6_var in enumerate(c6_vars):
    l_c6.append(ax_main.stairs(h_sig_c6[i].values() * fb_to_ab / xwidths, xbins, color=c6_color, linestyle=bsm_lines[i], linewidth=lw))
l_sm = ax_main.stairs(h_sig_sm.values() * fb_to_ab / xwidths, xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

l_sm.set_label('$\\mathrm{SM}$')
for i, cH_val in enumerate(cH_vars):
    sgn = '-' if cH_val < 0 else '+'
    l_c6[i].set_label(f'$C_{{H}} = {sgn}{abs(cH_val):.0f}$')
ax_main.legend(frameon=False, fontsize=12)

for i, c6_val in enumerate(c6_vars):
    ax_ratio.stairs(h_sig_c6[i].values() / h_sig_sm.values(), xbins, color=c6_color, linestyle=bsm_lines[i], linewidth=lw)
ax_ratio.stairs(h_sig_sm.values() / h_sig_sm.values(), xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

ax_main.set_yscale('linear')
ax_main.set_ylim(ymin_sig, ymax_sig)

ax_main.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax_main.transAxes, ha='left', va='top', fontsize=12)
ax_main.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_main.transAxes, ha='left', va='top', fontsize=12)
ax_ratio.set_ylim(-0.5, 2.5)
ax_ratio.yaxis.set_ticks([0.0, 1.0, 2.0])

ax_ratio.set_xscale('linear')
ax_ratio.set_xlim(xmin, xmax)
ax_ratio.set_xlabel(f'${xlabel}\\ \\mathrm{{[GeV]}}$', loc='right', fontsize=15)
ax_ratio.tick_params(top=True, labeltop=False)
if xname == 'mzz':
    ax_ratio.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax_main.set_ylabel(f'$d\sigma / d {xlabel}\\ \\mathrm{{[ab / GeV]}}$', loc='top', fontsize=15)
ax_ratio.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)

ax_main.tick_params(labelsize=12)
ax_ratio.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax_main_pos, ax_ratio_pos = ax_main.get_position(), ax_ratio.get_position()
ax_ratio.set_position([ax_main_pos.x0, ax_ratio_pos.y0, ax_main_pos.width, ax_ratio_pos.height]) # align 2nd x-axis with 1st

plt.savefig(f'sig_{xname}_c6.pdf')
plt.close()

In [ ]:
fig, (ax_main, ax_ratio) = plt.subplots(2,1, gridspec_kw={'height_ratios' : [3,1]}, figsize=(5,5), sharex=True, sharey=False)

l_c6 = []
for i, c6_var in enumerate(c6_vars):
    l_c6.append(ax_main.stairs(h_sbi_c6[i].values()/xwidths, xbins, color=c6_color, linestyle=bsm_lines[i], linewidth=lw))
l_sm = ax_main.stairs(h_sbi_sm.values()/xwidths, xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

l_sm.set_label('$\\mathrm{SM}$')
for i, cH_var in enumerate(cH_vars):
    sgn = '-' if cH_var < 0 else '+'
    l_c6[i].set_label(f'$C_{{H}} = {sgn}{abs(cH_var):.0f}$')
ax_main.legend(frameon=False, fontsize=12)

for i, c6_var in enumerate(c6_vars):
    ax_ratio.stairs(h_sbi_c6[i].values() / h_sbi_sm.values(), xbins, color=c6_color, linestyle=bsm_lines[i], linewidth=lw)

ax_ratio.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

ax_main.set_yscale('log')
ax_main.set_ylim(ymin_sbi, ymax_sbi)
# remove the first label
ax_main.set_yticks(np.logspace(np.log(ymin_sbi), np.log(ymax_sbi), int(np.log(ymax_sbi)-np.log(ymin_sbi)+1)))
ax_main_yticklabels = ax_main.get_yticklabels()
ax_main_yticklabels[0] = ""
ax_main.set_yticklabels(ax_main_yticklabels)

ax_main.text(0.04 ,0.12, '$gg \\rightarrow (h^{\\ast} \\rightarrow ) ZZ$', transform=ax_main.transAxes, ha='left', va='bottom', fontsize=12)
ax_main.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_main.transAxes, ha='left', va='bottom', fontsize=12)
ax_main.text(0.00 ,0.03, '$\\approx$', transform=ax_main.transAxes, ha='center', va='center', fontsize=12, bbox={'facecolor':'white', 'edgecolor':'none', 'pad':0.0})

ax_ratio.set_ylim(0.9,1.3)

ax_ratio.set_xscale('linear')
ax_ratio.set_xlim(xmin, xmax)
ax_ratio.tick_params(top=True, labeltop=False)
if xname == 'mzz':
    ax_ratio.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax_ratio.set_xlabel(f'${xlabel}\\ \\mathrm{{[GeV]}}$', loc='right', fontsize=15)

ax_main.set_ylabel(f'$d\sigma / d {xlabel}\\ \\mathrm{{[ab / GeV]}}$', loc='top', fontsize=15)
ax_ratio.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)

ax_main.tick_params(labelsize=12)
ax_ratio.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax_main_pos, ax_ratio_pos = ax_main.get_position(), ax_ratio.get_position()
ax_ratio.set_position([ax_main_pos.x0, ax_ratio_pos.y0, ax_main_pos.width, ax_ratio_pos.height]) # align 2nd x-axis with 1st

plt.savefig(f'sbi_{xname}_c6.pdf')
plt.close()

In [94]:
fig, (ax_main, ax_ratio) = plt.subplots(2,1, gridspec_kw={'height_ratios' : [3,1]}, figsize=(5,5), sharex=True, sharey=False)

# SIG

l_ct = []
for i, ct in enumerate(ct_vars):
    l_ct.append(ax_main.stairs(h_sig_ct[i].values() * fb_to_ab / xwidths, xbins, color=ct_color, linestyle=bsm_lines[i], linewidth=lw))
l_cg = []
for i, cg in enumerate(cg_vars):
    l_cg.append(ax_main.stairs(h_sig_cg[i].values() * fb_to_ab / xwidths, xbins, color=cg_color, linestyle=bsm_lines[i], linewidth=lw))
l_sm = ax_main.stairs(h_sig_sm.values() * fb_to_ab / xwidths, xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

l_sm.set_label('$\\mathrm{SM}$')
for i, ctH_val in enumerate(ctH_vars):
    sgn = '-' if ctH_val < 0 else '+'
    l_ct[i].set_label(f'$C_{{tH}} = {sgn}{abs(ctH_val):.5g}$')
for i, cHG_val in enumerate(cHG_vars):
    sgn = '-' if cHG_val < 0 else '+'
    l_cg[i].set_label(f'$C_{{HG}} = {sgn}{abs(cHG_val):.5g}$')

ax_main.legend(frameon=False, fontsize=12)

for i, ct_val in enumerate(ct_vars):
    ax_ratio.stairs(h_sig_ct[i].values() / h_sig_sm.values(), xbins, color=ct_color, linestyle=bsm_lines[i], linewidth=lw)
for i, cg_val in enumerate(cg_vars):
    ax_ratio.stairs(h_sig_cg[i].values() / h_sig_sm.values(), xbins, color=cg_color, linestyle=bsm_lines[i], linewidth=lw)
ax_ratio.stairs(h_sig_sm.values() / h_sig_sm.values(), xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

ax_main.set_yscale('linear')
ax_main.set_ylim(ymin_sig, ymax_sig)

ax_main.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax_main.transAxes, ha='left', va='top', fontsize=12)
ax_main.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_main.transAxes, ha='left', va='top', fontsize=12)
ax_ratio.set_ylim(-0.5, 2.5)
ax_ratio.yaxis.set_ticks([0.0, 1.0, 2.0])

ax_ratio.set_xscale('linear')
ax_ratio.set_xlim(xmin, xmax)
ax_ratio.set_xlabel(f'${xlabel}\\ \\mathrm{{[GeV]}}$', loc='right', fontsize=15)
ax_ratio.tick_params(top=True, labeltop=False)
if xname == 'mzz':
    ax_ratio.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax_main.set_ylabel(f'$d\sigma / d {xlabel}\\ \\mathrm{{[ab / GeV]}}$', loc='top', fontsize=15)
ax_ratio.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)

ax_main.tick_params(labelsize=12)
ax_ratio.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax_main_pos, ax_ratio_pos = ax_main.get_position(), ax_ratio.get_position()
ax_ratio.set_position([ax_main_pos.x0, ax_ratio_pos.y0, ax_main_pos.width, ax_ratio_pos.height]) # align 2nd x-axis with 1st

plt.savefig(f'sig_{xname}_ct_cg.pdf')
plt.close()

In [95]:
fig, (ax_main, ax_ratio) = plt.subplots(2,1, gridspec_kw={'height_ratios' : [3,1]}, figsize=(5,5), sharex=True, sharey=False)

l_ct = []
for i, ct in enumerate(ct_vars):
    l_ct.append(ax_main.stairs(h_sbi_ct[i].values() / xwidths, xbins, color=ct_color, linestyle=bsm_lines[i], linewidth=lw))
l_cg = []
for i, cg in enumerate(cg_vars):
    l_cg.append(ax_main.stairs(h_sbi_cg[i].values() / xwidths, xbins, color=cg_color, linestyle=bsm_lines[i], linewidth=lw))
l_sm = ax_main.stairs(h_sbi_sm.values() / xwidths, xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

l_sm.set_label('$\\mathrm{SM}$')
for i, ctH_val in enumerate(ctH_vars):
    sgn = '-' if ctH_val < 0 else '+'
    l_ct[i].set_label(f'$C_{{tH}} = {sgn}{abs(ctH_val):.5g}$')
for i, cHG_val in enumerate(cHG_vars):
    sgn = '-' if cHG_val < 0 else '+'
    l_cg[i].set_label(f'$C_{{HG}} = {sgn}{abs(cHG_val):.5g}$')

ax_main.legend(frameon=False, fontsize=12)

for i, ct_var in enumerate(ct_vars):
    ax_ratio.stairs(h_sbi_ct[i].values() / h_sbi_sm.values(), xbins, color=ct_color, linestyle=bsm_lines[i], linewidth=lw)
for i, cg_var in enumerate(cg_vars):
    ax_ratio.stairs(h_sbi_cg[i].values() / h_sbi_sm.values(), xbins, color=cg_color, linestyle=bsm_lines[i], linewidth=lw)

ax_ratio.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=sm_color, linestyle=sm_line, linewidth=lw)

ax_main.set_yscale('log')
ax_main.set_ylim(ymin_sbi, ymax_sbi)
# remove the first label
ax_main_yticklabels = ax_main.get_yticklabels()
ax_main_yticklabels[0] = ""
ax_main.set_yticklabels(ax_main_yticklabels)

ax_main.text(0.04 ,0.12, '$gg \\rightarrow (h^{\\ast} \\rightarrow ) ZZ$', transform=ax_main.transAxes, ha='left', va='bottom', fontsize=12)
ax_main.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_main.transAxes, ha='left', va='bottom', fontsize=12)
ax_main.text(0.00 ,0.03, '$\\approx$', transform=ax_main.transAxes, ha='center', va='center', fontsize=12, bbox={'facecolor':'white', 'edgecolor':'none', 'pad':0.0})

ax_ratio.set_ylim(0.9,1.3)

ax_ratio.set_xscale('linear')
ax_ratio.set_xlim(xmin, xmax)
ax_ratio.tick_params(top=True, labeltop=False)
if xname == 'mzz':
    ax_ratio.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax_ratio.set_xlabel(f'${xlabel}\\ \\mathrm{{[GeV]}}$', loc='right', fontsize=15)

ax_main.set_ylabel(f'$d\sigma / d {xlabel}\\ \\mathrm{{[ab / GeV]}}$', loc='top', fontsize=15)
ax_ratio.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)

ax_main.tick_params(labelsize=12)
ax_ratio.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax_main_pos, ax_ratio_pos = ax_main.get_position(), ax_ratio.get_position()
ax_ratio.set_position([ax_main_pos.x0, ax_ratio_pos.y0, ax_main_pos.width, ax_ratio_pos.height]) # align 2nd x-axis with 1st

plt.savefig(f'sbi_{xname}_ct_cg.pdf')
plt.close()

/tmp/ipykernel_1821371/2463024071.py:33: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_main.set_yticklabels(ax_main_yticklabels)
